In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
from scipy.stats import pearsonr, spearmanr

In [2]:
class protT5Dataset(Dataset):
    def __init__(self, embeddings, scores):
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.scores = torch.tensor(scores, dtype=torch.float32)

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.scores[idx]

In [3]:
def load_data_with_embeddings (csv_path, embeddings_path, indices_path, batch_size=32):
    df = pd.read_csv(csv_path)

    assert "split" in df.columns, "CSV does not contain 'split' column"

    print("Split distribution:")
    print(df["split"].value_counts().to_string())
    print("\n Proteins per split:")
    print(df.groupby("split")["pdb_fn"].nunique().to_string())

    embeddings = np.load(embeddings_path, mmap_mode="r")
    print("Embeddings shape: ", embeddings.shape)

    embedding_indices = np.load(indices_path, mmap_mode="r")
    print("Embeddings indices shape: ", embedding_indices.shape)


    if len(embeddings) != len(embedding_indices):
        raise ValueError("Embeddings and indices do not match")

    idx_to_pos = {}
    for pos, original_idx in enumerate(embedding_indices):
        int_id = int(original_idx)
        idx_to_pos[int_id] = pos

    def get_split_data(split_name):
        split_df = df[df["split"] == split_name].copy()
        positions = [idx_to_pos[int(i)] for i in split_df["original_index"].values]
        return embeddings[positions], split_df["total_score_z"].to_numpy()

    train_embeddings, train_scores = get_split_data("train")
    val_embeddings, val_scores = get_split_data("val")
    test_embeddings, test_scores = get_split_data("test")

    train_loader = DataLoader(protT5Dataset(train_embeddings, train_scores), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(protT5Dataset(val_embeddings, val_scores), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(protT5Dataset(test_embeddings, test_scores), batch_size=batch_size, shuffle=False)

    print("Train: ", len(train_loader))
    print("Val: ", len(val_loader))
    print("Test: ", len(test_loader))

    return train_loader, val_loader, test_loader, embeddings.shape[1]

In [4]:
def calc_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    pearson, _ = pearsonr(y_true, y_pred) #otherwise returns tuple of coefficient and p-value
    spearman, _ = spearmanr(y_true, y_pred)

    return {
        "MSE": mse, "MAE": mae, "RMSE": rmse,
        "R2": r2, "Pearson": pearson, "Spearman": spearman
    }

In [5]:
def evaluate(model, loader, criterion):
    model.eval()
    all_preds=[]
    all_targets=[]
    total_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            y_pred = model(x_batch).squeeze(-1)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()

            all_preds.extend(y_pred.tolist())
            all_targets.extend(y_batch.tolist())

    metrics = calc_metrics(all_targets, all_preds)
    metrics["loss"] = total_loss / len(loader)
    return metrics

In [6]:
def train(model, train_loader, optimizer, criterion, epochs = 10, val_loader=None):
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(x_batch).squeeze(-1)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        print("Epoch: ", epoch+1,"/",epochs, "| Train Loss: ", avg_train_loss)

        if val_loader is not None:
            val_metrics = evaluate(model, val_loader, criterion)
            print("Validation Metrics: ", val_metrics)
            print("-"*30)



In [7]:
if __name__ == "__main__":
    csv_path = r"C:\Users\paulb\PycharmProjects\METL_GLOBAL\data\data_split_normalized_combined.csv"
    embedding_path = r"C:\Users\paulb\PycharmProjects\METL_GLOBAL\data\prot_t5_embeddings_combined.npy"
    indices_path = r"C:\Users\paulb\PycharmProjects\METL_GLOBAL\data\prot_t5_indices_combined.npy"
    BATCH_SIZE = 32
    EPOCHS = 10
    LEARNING_RATE = 0.001

    train_loader, val_loader, test_loader, input_dim = load_data_with_embeddings(csv_path=csv_path, embeddings_path=embedding_path,indices_path=indices_path, batch_size=BATCH_SIZE )

    model = nn.Linear(in_features=input_dim, out_features=1)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train(model=model, train_loader=train_loader, optimizer=optimizer, criterion=criterion, epochs=EPOCHS,val_loader=val_loader)

    print("Training complete! | Running on Test set:")

    test_metrics = evaluate(model=model, loader=test_loader, criterion=criterion)

    print("-" * 50)
    print(f"Test Loss (MSE): {test_metrics['loss']:.4f}")
    print(f"Test R2 Score:   {test_metrics['R2']:.4f}")
    print(f"Test Pearson r:  {test_metrics['Pearson']:.4f}")
    print(f"Test Spearman r: {test_metrics['Spearman']:.4f}")
    print("=" * 50)


Split distribution:
split
train    112867
val       14393
test      14391

 Proteins per split:
split
test      15
train    118
val       15
Embeddings shape:  (141651, 1024)
Embeddings indices shape:  (141651,)
Train:  3528
Val:  450
Test:  450
Epoch:  1 / 10 | Train Loss:  0.9524422410554558
Validation Metrics:  {'MSE': 0.9750678258582284, 'MAE': 0.7685096403780385, 'RMSE': np.float64(0.9874552272676612), 'R2': 0.0187578526059613, 'Pearson': np.float64(0.16377953815576696), 'Spearman': np.float64(0.13670895183110363), 'loss': 0.9752092005146874}
------------------------------
Epoch:  2 / 10 | Train Loss:  0.9295383843840385
Validation Metrics:  {'MSE': 0.9939353029182063, 'MAE': 0.7730793174531099, 'RMSE': np.float64(0.9969630398957658), 'R2': -0.00022909703515261448, 'Pearson': np.float64(0.14930516517168277), 'Spearman': np.float64(0.1293599792392516), 'loss': 0.9940062250031365}
------------------------------
Epoch:  3 / 10 | Train Loss:  0.920114687688294
Validation Metrics:  {'M